# Experiment 4: Best Model 1 — Focused Hyperparameter Tuning

**Objective:** Load `best_model_1` from Experiment 3 and perform focused tuning using `RandomizedSearchCV` (20 iterations, 5-Fold CV, RMSE).

**Outputs:** Learning curve, Validation curve, Feature importance, `best_model_1_tuned.pkl`

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import json
warnings.filterwarnings('ignore')

from sklearn import set_config
set_config(transform_output='pandas')

from sklearn.model_selection import (train_test_split, RandomizedSearchCV,
                                     learning_curve, validation_curve)
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import clean_data_utils
print('Imports successful')

## 2. Load Data & Recreate Split

In [ ]:
raw_df = pd.read_csv('food_delivery_data.csv')
clean_data_utils.perform_data_cleaning(raw_df, saved_data_path='cleaned_data.csv')
df = pd.read_csv('cleaned_data.csv')
df.drop(columns=['rider_id','restaurant_latitude','restaurant_longitude',
                 'delivery_latitude','delivery_longitude','order_day','order_time_hour'],
        inplace=True)

num_col          = ['age','ratings','pickup_time_minutes','distance']
nominal_cat_cols = ['type_of_order','type_of_vehicle','festival','city_type',
                    'city_name','weather','order_day_of_week','order_time_of_day']
ordinal_cat_cols = ['distance_type','traffic']
distance_type_order = ['short','very_long','medium','long']
traffic_order       = ['low','medium','high','jam']
features_to_fill_mode    = ['multiple_deliveries','festival','city_name']
features_to_fill_missing = [c for c in nominal_cat_cols if c not in features_to_fill_mode]

print(f'Data loaded: {df.shape}')

## 3. Load Best Model 1 Metadata

In [ ]:
with open('best_models_meta.json') as f:
    meta = json.load(f)

m1 = meta['best_model_1']
MODEL_NAME = m1['name']
DATASET    = m1['dataset']

print(f'Best Model 1: {MODEL_NAME} trained on {DATASET}')
print(f'Exp 3 RMSE: {m1["rmse"]:.4f}  R2: {m1["r2"]:.4f}')

## 4. Recreate Dataset & Preprocessor

In [ ]:
def build_imputed_preprocessor():
    simple_imp = ColumnTransformer([
        ('mode_imp',    SimpleImputer(strategy='most_frequent'),         features_to_fill_mode),
        ('missing_imp', SimpleImputer(strategy='constant',fill_value='missing'), features_to_fill_missing)
    ], remainder='passthrough', n_jobs=-1, force_int_remainder_cols=False, verbose_feature_names_out=False)
    encoder = ColumnTransformer([
        ('scale',   MinMaxScaler(), num_col),
        ('nominal', OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False), nominal_cat_cols),
        ('ordinal', OrdinalEncoder(categories=[distance_type_order,traffic_order],
                                   encoded_missing_value=-999, handle_unknown='use_encoded_value',
                                   unknown_value=-1), ordinal_cat_cols)
    ], remainder='passthrough', n_jobs=-1, force_int_remainder_cols=False, verbose_feature_names_out=False)
    return Pipeline([('simple_imp',simple_imp),('encoder',encoder),('knn_imp',KNNImputer(n_neighbors=5))])

def build_clean_preprocessor():
    return ColumnTransformer([
        ('scale',   MinMaxScaler(), num_col),
        ('nominal', OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False), nominal_cat_cols),
        ('ordinal', OrdinalEncoder(categories=[distance_type_order,traffic_order]), ordinal_cat_cols)
    ], remainder='passthrough', n_jobs=-1, force_int_remainder_cols=False, verbose_feature_names_out=False)

if DATASET == 'Dataset_A':
    df_use = df.dropna().copy()
    preprocessor = build_clean_preprocessor()
else:
    df_use = df.copy()
    preprocessor = build_imputed_preprocessor()

X = df_use.drop(columns='time_taken')
y = df_use['time_taken']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pt = PowerTransformer(method='yeo-johnson')
y_train_pt = pt.fit_transform(y_train.values.reshape(-1,1))

print(f'Train:{X_train.shape}  Test:{X_test.shape}')
print(f'Dataset used: {DATASET}')

## 5. Define Tuning Grid for Best Model 1

In [ ]:
def get_model_and_grid(name):
    if name == 'Random Forest':
        return RandomForestRegressor(random_state=42, n_jobs=-1), {
            'model__n_estimators':      [200,300,400,500],
            'model__max_depth':         [None,10,15,20,25],
            'model__min_samples_split': [2,4,6],
            'model__min_samples_leaf':  [1,2,4],
            'model__max_features':      ['sqrt','log2',0.5]
        }
    elif name == 'Gradient Boosting':
        return GradientBoostingRegressor(random_state=42), {
            'model__n_estimators':  [150,200,300,400],
            'model__learning_rate': [0.03,0.05,0.08,0.1,0.15],
            'model__max_depth':     [3,4,5,6],
            'model__subsample':     [0.7,0.8,0.9,1.0],
            'model__min_samples_split': [2,5,10]
        }
    elif name == 'XGBoost':
        return XGBRegressor(random_state=42, n_jobs=-1, verbosity=0), {
            'model__n_estimators':     [200,300,400,500],
            'model__learning_rate':    [0.03,0.05,0.08,0.1,0.15],
            'model__max_depth':        [3,4,5,6,7],
            'model__subsample':        [0.7,0.8,0.9,1.0],
            'model__colsample_bytree': [0.7,0.8,0.9,1.0],
            'model__reg_alpha':        [0,0.1,0.5],
            'model__reg_lambda':       [1,2,5]
        }
    elif name == 'LightGBM':
        return LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1), {
            'model__n_estimators':  [200,300,400,500],
            'model__learning_rate': [0.03,0.05,0.08,0.1,0.15],
            'model__num_leaves':    [31,63,127],
            'model__max_depth':     [-1,5,8,10],
            'model__subsample':     [0.7,0.8,0.9,1.0],
            'model__reg_alpha':     [0,0.1,0.5],
            'model__reg_lambda':    [1,2,5]
        }
    else:
        raise ValueError(f'Unknown model: {name}')

model, param_grid = get_model_and_grid(MODEL_NAME)
print(f'Tuning grid for: {MODEL_NAME}')
print(f'Parameters: {list(param_grid.keys())}')

## 6. RandomizedSearchCV — 20 iterations, 5-Fold CV

In [ ]:
neg_rmse = make_scorer(lambda y, yp: -np.sqrt(mean_squared_error(y, yp)))

pipeline = Pipeline([('preprocessing', preprocessor), ('model', model)])

search = RandomizedSearchCV(
    pipeline, param_distributions=param_grid,
    n_iter=20, cv=5, scoring=neg_rmse,
    n_jobs=-1, random_state=42, refit=True,
    return_train_score=True, verbose=1
)
search.fit(X_train, np.asarray(y_train_pt).ravel())

best_pipeline = search.best_estimator_
print(f'\nBest CV RMSE (transformed space): {-search.best_score_:.4f}')
print(f'Best Parameters: {search.best_params_}')

## 7. Evaluate on Test Set

In [ ]:
y_pred_pt   = best_pipeline.predict(X_test)
y_pred_orig = pt.inverse_transform(y_pred_pt.reshape(-1,1)).ravel()

rmse = np.sqrt(mean_squared_error(y_test, y_pred_orig))
mae  = mean_absolute_error(y_test, y_pred_orig)
r2   = r2_score(y_test, y_pred_orig)
mape = np.mean(np.abs((y_test.values - y_pred_orig) / np.maximum(np.abs(y_test.values),1e-8)))*100

print(f'\n=== {MODEL_NAME} Tuned — Test Results ===')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R2   : {r2:.4f}')
print(f'MAPE : {mape:.2f}%')
print(f'\nImprovement vs Exp 3: RMSE {m1["rmse"]:.4f} → {rmse:.4f}')

tuned_metrics = {'RMSE':round(rmse,4),'MAE':round(mae,4),'R2':round(r2,4),'MAPE':round(mape,4),
                 'Best Parameters': str(search.best_params_)}
pd.DataFrame([tuned_metrics]).to_csv('exp4_tuned_metrics.csv', index=False)
print('Saved: exp4_tuned_metrics.csv')

## 8. Cross-Validation Results

In [ ]:
cv_results = pd.DataFrame(search.cv_results_)
cv_top = cv_results.sort_values('mean_test_score', ascending=False)[[
    'mean_test_score','std_test_score','mean_train_score','rank_test_score'
]].head(10)
cv_top['mean_test_score']  = -cv_top['mean_test_score']   # convert to positive RMSE
cv_top['mean_train_score'] = -cv_top['mean_train_score']
cv_top.columns = ['CV RMSE (val)','Std','CV RMSE (train)','Rank']
print('Top 10 CV Combinations:')
print(cv_top.to_string(index=False))

## 9. Visualizations

In [ ]:
# Actual vs Predicted + Residual
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'{MODEL_NAME} Tuned — Predictions', fontsize=13, fontweight='bold')

ax1.scatter(y_test, y_pred_orig, alpha=0.35, s=10, color='steelblue')
mn, mx = min(y_test.min(), y_pred_orig.min()), max(y_test.max(), y_pred_orig.max())
ax1.plot([mn,mx],[mn,mx],'r--',lw=2); ax1.set_xlabel('Actual'); ax1.set_ylabel('Predicted')
ax1.set_title('Actual vs Predicted'); ax1.grid(alpha=0.3)

resid = y_test.values - y_pred_orig
ax2.scatter(y_pred_orig, resid, alpha=0.35, s=10, color='coral')
ax2.axhline(0,color='black',lw=2,ls='--'); ax2.set_xlabel('Predicted'); ax2.set_ylabel('Residual')
ax2.set_title('Residual Plot'); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('exp4_pred_residual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exp4_pred_residual.png')

In [ ]:
# Learning Curve
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    best_pipeline, X_train, np.asarray(y_train_pt).ravel(),
    cv=5, scoring=neg_rmse, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 8)
)

train_rmse = -train_scores
val_rmse   = -val_scores

plt.figure(figsize=(10, 5))
plt.plot(train_sizes, train_rmse.mean(axis=1), 'o-', color='steelblue', label='Train RMSE')
plt.fill_between(train_sizes,
                 train_rmse.mean(1)-train_rmse.std(1),
                 train_rmse.mean(1)+train_rmse.std(1), alpha=0.15, color='steelblue')
plt.plot(train_sizes, val_rmse.mean(axis=1), 'o-', color='coral', label='Val RMSE')
plt.fill_between(train_sizes,
                 val_rmse.mean(1)-val_rmse.std(1),
                 val_rmse.mean(1)+val_rmse.std(1), alpha=0.15, color='coral')
plt.xlabel('Training Set Size'); plt.ylabel('RMSE (transformed space)')
plt.title(f'Learning Curve — {MODEL_NAME} Tuned', fontweight='bold')
plt.legend(); plt.grid(alpha=0.35)
plt.tight_layout()
plt.savefig('exp4_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exp4_learning_curve.png')

In [ ]:
# Validation Curve — tune most important hyperparameter
from sklearn.model_selection import validation_curve

param_name = 'model__n_estimators'
param_range = [50, 100, 150, 200, 300, 400]

train_vc, val_vc = validation_curve(
    Pipeline([('preprocessing', build_clean_preprocessor() if DATASET=='Dataset_A' else build_imputed_preprocessor()),
              ('model', get_model_and_grid(MODEL_NAME)[0])]),
    X_train, np.asarray(y_train_pt).ravel(),
    param_name=param_name, param_range=param_range,
    cv=5, scoring=neg_rmse, n_jobs=-1
)

plt.figure(figsize=(9, 5))
plt.plot(param_range, (-train_vc).mean(1), 'o-', color='steelblue', label='Train RMSE')
plt.plot(param_range, (-val_vc).mean(1),   'o-', color='coral',     label='Val RMSE')
plt.fill_between(param_range, (-val_vc).mean(1)-(-val_vc).std(1),
                              (-val_vc).mean(1)+(-val_vc).std(1), alpha=0.15, color='coral')
plt.xlabel('n_estimators'); plt.ylabel('RMSE (transformed space)')
plt.title(f'Validation Curve — {MODEL_NAME}', fontweight='bold')
plt.legend(); plt.grid(alpha=0.35)
plt.tight_layout()
plt.savefig('exp4_validation_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exp4_validation_curve.png')

In [ ]:
# Feature Importance
model_step = best_pipeline.named_steps['model']
if hasattr(model_step, 'feature_importances_'):
    imp = model_step.feature_importances_
    try:
        names = best_pipeline.named_steps['preprocessing'].get_feature_names_out()
    except Exception:
        names = [f'f{i}' for i in range(len(imp))]
    s = pd.Series(imp, index=names[:len(imp)]).sort_values(ascending=False).head(20)
    plt.figure(figsize=(10,6))
    sns.barplot(x=s.values, y=s.index, palette='viridis')
    plt.title(f'Top 20 Feature Importances — {MODEL_NAME} Tuned', fontweight='bold')
    plt.xlabel('Importance'); plt.tight_layout()
    plt.savefig('exp4_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: exp4_feature_importance.png')

## 10. Save Tuned Model

In [ ]:
joblib.dump(best_pipeline, 'best_model_1_tuned.pkl')
# Also save pt for inverse_transform in Exp 6
joblib.dump(pt, 'pt_model_1.pkl')
joblib.dump({'X_test': X_test, 'y_test': y_test}, 'exp4_test_data.pkl')

print('='*50)
print('EXPERIMENT 4 COMPLETE')
print('='*50)
print(f'Model: {MODEL_NAME} ({DATASET})')
print(f'RMSE: {rmse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}  MAPE: {mape:.2f}%')
print('Saved: best_model_1_tuned.pkl')